In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Prerequisites ===
# Conda environment ir-class for IR classification, registered as a Jupyter kernel.
"""
conda create -n ir-class python=3.10 -y
conda activate ir-class

conda install -y -c conda-forge \
  ipykernel numpy pandas matplotlib seaborn scikit-learn

pip install --no-cache-dir joblib

python -m ipykernel install --user --name ir-class --display-name "ir-class"
"""

In [ ]:
# === Cell 3: Imports ===

import os
import re
import sys
import json
import random
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.decomposition   import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.base            import clone
from sklearn.metrics         import (
    f1_score, balanced_accuracy_score, precision_score,
    recall_score, accuracy_score, confusion_matrix,
)
import joblib

from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"]      = 120
plt.rcParams["savefig.dpi"]     = 150
plt.rcParams["figure.figsize"]  = (8, 5)
plt.rcParams["axes.titlesize"]  = 12
plt.rcParams["axes.labelsize"]  = 11
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

In [ ]:
# === Cell 4: Config ===

# --- Input paths ---
SPECTRA_DIR      = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/01_Preprocessing_Investigation/C8_als_sg-2nd-deriv")
OBJECT_SPLIT_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export/object_split.csv")
GROUND_TRUTH_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/spectra_ground_truth_v7.csv")

# --- Output paths ---
BENCHMARK_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/04_IR_Classification/20260616_Classifier_Benchmark_Pre_Processing_Influence/C8_als_sg-2nd-deriv")
RESULTS_DIR    = BENCHMARK_ROOT / "results"
MODELS_DIR     = RESULTS_DIR / "models"
for p in [BENCHMARK_ROOT, RESULTS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_NAME = BENCHMARK_ROOT.name
RANDOM_SEED  = 42

# Set to True to allow re-running into an existing results directory
# (checked in the Sanity Check cell).
ALLOW_OVERWRITE = False

# --- Dataset semantics ---
# Spectrum filename: <PREFIX_MATERIAL>_obj_<OBJECT_ID>_<REGION> (no extension).
# The prefix is only a naming convention; the true class of each region comes
# from the ground-truth CSV (e.g. an HDPE bottle can have a Paper label).
FILENAME_REGEX = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_id>\d+)_(?P<region>body|cap|label)$",
    re.IGNORECASE,
)
OBJECT_ID_WIDTH = 2                         # zero-pad object ids to this width
VALID_REGIONS   = ["body", "cap", "label"]
VALID_SPLITS    = {"cv", "test"}            # "skipped" objects are excluded

# Ground-truth column holding the true class for each region.
REGION_TO_GT_COLUMN = {
    "body":  "body_class",
    "label": "label_class",
    "cap":   "cap_class",
}

# --- Outer split: objects not in the segmentation split ---
# Objects missing from object_split.csv go to the CV pool, except those listed
# here, which go to the test set (~21 % of objects, as in the segmentation split).
NEW_TEST_OBJECTS = [
    ("HDPE", 13),
    ("PET",  16), ("PET", 21), ("PET", 24), ("PET", 28),
    ("PS",   14), ("PS",  18),
]

# --- Cross-validation ---
NUM_FOLDS = 4
N_JOBS    = -1   # CPU cores for GridSearchCV

# --- Evaluation metrics ---
# macro-F1 is primary: it weights all polymer classes equally, including rare ones.
PRIMARY_METRIC    = "macro-F1"
SECONDARY_METRICS = ["balanced_accuracy", "macro-precision", "macro-recall", "accuracy"]

# --- PCA dimensionality search space ---
# Pipeline: scaler -> pca -> clf; PCA is fit on each fold's training spectra only.
# None keeps all components (min(n_train, n_features)). Values must stay below
# the per-fold training-sample count.
PCA_N_COMPONENTS_GRID = [5, 10, 20, 30, None]

# --- Classifier hyperparameter grids ---
# Each entry: (estimator, param_grid) with Pipeline prefix "clf__";
# "pca__n_components" is added to every grid below. Grids are saved to RESULTS_DIR.
# Spectra are already z-score normalised per sample by the preprocessing.

_CLASSIFIER_GRIDS = {
    "SVM": (
        SVC(probability=True, random_state=RANDOM_SEED, max_iter=10_000),
        [
            {"clf__kernel": ["linear"], "clf__C": [0.1, 1, 10, 100]},
            {"clf__kernel": ["rbf"],    "clf__C": [0.1, 1, 10, 100],
             "clf__gamma":  ["scale", 0.001, 0.01, 0.1]},
        ],
    ),
    "kNN": (
        KNeighborsClassifier(algorithm="auto"),
        [
            {"clf__n_neighbors": [1, 3, 5, 7, 9], "clf__weights": ["uniform", "distance"],
             "clf__metric": ["euclidean"]},
            {"clf__n_neighbors": [1, 3, 5, 7, 9], "clf__weights": ["uniform", "distance"],
             "clf__metric": ["cosine"], "clf__algorithm": ["brute"]},
        ],
    ),
    "RF": (
        RandomForestClassifier(random_state=RANDOM_SEED),
        {"clf__n_estimators": [200, 500], "clf__max_depth": [None, 5, 10, 20],
         "clf__min_samples_leaf": [1, 2, 5]},
    ),
    "DT": (
        DecisionTreeClassifier(criterion="gini", random_state=RANDOM_SEED),
        {"clf__max_depth": [None, 3, 5, 10, 20], "clf__min_samples_leaf": [1, 2, 5]},
    ),
    "LR": (
        # penalty defaults to "l2"; multinomial loss is automatic for lbfgs/saga.
        # penalty= and multi_class= are not set (deprecated in sklearn 1.5, removed in 1.7).
        LogisticRegression(max_iter=5_000, random_state=RANDOM_SEED),
        {"clf__C": [0.01, 0.1, 1, 10, 100], "clf__solver": ["lbfgs", "saga"]},
    ),
}

def _inject_pca_grid(param_grid):
    """Add pca__n_components to every sub-grid so PCA is tuned with the classifier."""
    if isinstance(param_grid, list):
        return [{**sub, "pca__n_components": PCA_N_COMPONENTS_GRID} for sub in param_grid]
    return {**param_grid, "pca__n_components": PCA_N_COMPONENTS_GRID}

PARAM_GRIDS = {
    name: (est, _inject_pca_grid(grid))
    for name, (est, grid) in _CLASSIFIER_GRIDS.items()
}

# --- Output artefact paths ---
SAMPLE_TABLE_CSV     = RESULTS_DIR / f"{PROJECT_NAME}_sample_table.csv"
FOLD_ASSIGNMENT_CSV  = RESULTS_DIR / f"{PROJECT_NAME}_fold_assignments.csv"
FOLD_REASSIGN_CSV    = RESULTS_DIR / f"{PROJECT_NAME}_fold_reassignments.csv"
SPLIT_OBJECTS_CSV    = RESULTS_DIR / f"{PROJECT_NAME}_cv_split_objects.csv"
SPLIT_SUMMARY_CSV    = RESULTS_DIR / f"{PROJECT_NAME}_split_summary.csv"
HPO_RESULTS_CSV      = RESULTS_DIR / f"{PROJECT_NAME}_hpo_results.csv"
CV_METRICS_CSV       = RESULTS_DIR / f"{PROJECT_NAME}_cv_metrics.csv"
CV_PER_CLASS_CSV     = RESULTS_DIR / f"{PROJECT_NAME}_cv_per_class_metrics.csv"
TEST_METRICS_CSV     = RESULTS_DIR / f"{PROJECT_NAME}_test_metrics.csv"
TEST_PREDICTIONS_CSV = RESULTS_DIR / f"{PROJECT_NAME}_test_predictions.csv"
PARAM_GRIDS_JSON     = RESULTS_DIR / f"{PROJECT_NAME}_param_grids.json"
CONFUSION_MATRIX_PNG = RESULTS_DIR / f"{PROJECT_NAME}_confusion_matrices.png"
CV_COMPARISON_PNG    = RESULTS_DIR / f"{PROJECT_NAME}_cv_comparison.png"
TEST_COMPARISON_PNG  = RESULTS_DIR / f"{PROJECT_NAME}_test_comparison.png"

# --- Sanity printout ---
print("Configuration")
print("-" * 80)
print("SPECTRA_DIR:         ", SPECTRA_DIR)
print("OBJECT_SPLIT_CSV:    ", OBJECT_SPLIT_CSV)
print("GROUND_TRUTH_CSV:    ", GROUND_TRUTH_CSV)
print("BENCHMARK_ROOT:      ", BENCHMARK_ROOT)
print("RESULTS_DIR:         ", RESULTS_DIR)
print("MODELS_DIR:          ", MODELS_DIR)
print("PROJECT_NAME:        ", PROJECT_NAME)
print("RANDOM_SEED:         ", RANDOM_SEED)
print("NUM_FOLDS:           ", NUM_FOLDS)
print("N_JOBS:              ", N_JOBS, "(CPU; sklearn classifiers have no GPU path)")
print("PRIMARY_METRIC:      ", PRIMARY_METRIC)
print("PCA_N_COMPONENTS_GRID:", PCA_N_COMPONENTS_GRID)
print("Classifiers:         ", list(PARAM_GRIDS.keys()))

In [ ]:
# === Cell 5: Sanity Check ===

# Stop if a previous run's artefacts exist and ALLOW_OVERWRITE is False.
_existing = [p for p in [CV_METRICS_CSV, TEST_METRICS_CSV] if p.exists()]
if _existing and not ALLOW_OVERWRITE:
    raise RuntimeError(
        "Existing benchmark artefacts found in RESULTS_DIR and ALLOW_OVERWRITE is "
        f"False:\n  " + "\n  ".join(str(p) for p in _existing) +
        "\nSet ALLOW_OVERWRITE = True in Cell 4 to re-run into this directory."
    )

import sklearn
print("Python:       ", sys.version.split()[0])
print("numpy:        ", np.__version__)
print("pandas:       ", pd.__version__)
print("scikit-learn: ", sklearn.__version__)

PATHS_TO_CHECK = [
    ("SPECTRA_DIR",      SPECTRA_DIR),
    ("OBJECT_SPLIT_CSV", OBJECT_SPLIT_CSV),
    ("GROUND_TRUTH_CSV", GROUND_TRUTH_CSV),
    ("BENCHMARK_ROOT",   BENCHMARK_ROOT),
    ("RESULTS_DIR",      RESULTS_DIR),
]
all_ok = True
print("\nPath checks")
for name, path in PATHS_TO_CHECK:
    kind   = "DIR " if path.is_dir()  else "FILE"
    status = "OK     " if path.exists() else "MISSING"
    print(f"  [{kind}] {status}  {name}: {path}")
    if not path.exists():
        all_ok = False

def read_spectrum_file(path):
    """Load a semicolon-delimited wavenumber;reflectance file, return sorted DataFrame."""
    df = pd.read_csv(path, sep=";", header=None,
                     names=["wavenumber", "reflectance"], engine="python")
    if df.shape[1] != 2:
        raise ValueError(f"Expected 2 columns, got {df.shape[1]}: {path}")
    df["wavenumber"]  = pd.to_numeric(df["wavenumber"],  errors="coerce")
    df["reflectance"] = pd.to_numeric(df["reflectance"], errors="coerce")
    if df.isna().any().any():
        raise ValueError(f"Non-numeric values in {path}")
    if df.empty:
        raise ValueError(f"Empty file: {path}")
    return df.sort_values("wavenumber").reset_index(drop=True)

sample_files = [p for p in SPECTRA_DIR.iterdir() if p.is_file()][:3] if SPECTRA_DIR.exists() else []
print(f"\nSpectrum smoke test (first {len(sample_files)} files):")
for p in sample_files:
    try:
        df = read_spectrum_file(p)
        print(f"  OK  {p.name}  ({len(df)} points, "
              f"wn=[{df['wavenumber'].min():.1f}, {df['wavenumber'].max():.1f}])")
    except Exception as exc:
        print(f"  FAIL {p.name}: {exc}")
        all_ok = False

if not all_ok:
    raise RuntimeError("Sanity check FAILED — fix paths before continuing.")
print("\nSanity check PASS")

In [ ]:
# === Cell 6: File Discovery and Parsing ===

# Files are extensionless; filter by FILENAME_REGEX on the stem.
# object_stem (e.g. "HDPE_obj_01") is the join key to the ground-truth CSV;
# group_id has the same value and is the object-level unit for splitting.

def parse_filename(path):
    """Extract material, object_id, region, object_stem, group_id from a filename."""
    match = FILENAME_REGEX.match(Path(path).stem)
    if not match:
        return None
    d   = match.groupdict()
    mat = d["material"].upper()
    oid = d["object_id"].zfill(OBJECT_ID_WIDTH)
    stem = f"{mat}_obj_{oid}"
    return {
        "path":            str(path),
        "filename":        Path(path).name,
        "prefix_material": mat,           # object naming prefix (NOT the label)
        "object_id":       oid,
        "region":          d["region"].lower(),
        "object_stem":     stem,          # join key to ground truth
        "group_id":        stem,          # object-level grouping unit
    }

all_files     = sorted([p for p in SPECTRA_DIR.iterdir() if p.is_file()])
parsed_rows   = []
skipped_files = []
for p in all_files:
    res = parse_filename(p)
    (parsed_rows if res is not None else skipped_files).append(res if res is not None else p.name)

if skipped_files:
    print(f"WARNING: {len(skipped_files)} files did not match FILENAME_REGEX and are excluded:")
    for name in skipped_files[:10]:
        print(f"  - {name}")
    if len(skipped_files) > 10:
        print(f"  ... and {len(skipped_files) - 10} more")

raw_meta_df = pd.DataFrame(parsed_rows)
if raw_meta_df.empty:
    raise RuntimeError(f"No files matched FILENAME_REGEX in {SPECTRA_DIR}.")

print(f"Files discovered:  {len(all_files)}")
print(f"Files parsed:      {len(raw_meta_df)}")
print(f"Unique prefixes:   {sorted(raw_meta_df['prefix_material'].unique())}")
print(f"Unique objects:    {raw_meta_df['group_id'].nunique()}")
print(f"Unique regions:    {sorted(raw_meta_df['region'].unique())}")

print("\nSpectra per region:")
display(raw_meta_df.groupby("region").size().rename("n_spectra"))

In [ ]:
# === Cell 7: Ground Truth and Outer Split ===

# --- (a) Region-specific ground truth ---
# One row per object with the true class of body / label / cap. An empty class
# means that component was excluded from the balanced dataset.
gt_df = pd.read_csv(GROUND_TRUTH_CSV, sep=";", dtype=str)
gt_df.columns = [c.strip() for c in gt_df.columns]
for c in gt_df.columns:
    gt_df[c] = gt_df[c].str.strip().replace({"": np.nan})

print("Ground-truth columns:", list(gt_df.columns))
print(f"Ground-truth objects: {len(gt_df)}")
for region, col in REGION_TO_GT_COLUMN.items():
    vals = sorted(gt_df[col].dropna().unique())
    print(f"  {region:5s} ({col}): classes={vals}  missing={gt_df[col].isna().sum()}")

gt_indexed = gt_df.set_index("filename")

# --- (b) Assign region-specific true class ---
def lookup_true_class(row):
    """Return the ground-truth polymer class for this spectrum's region, or NaN."""
    stem = row["object_stem"]
    col  = REGION_TO_GT_COLUMN[row["region"]]
    if stem not in gt_indexed.index:
        return np.nan
    return gt_indexed.loc[stem, col]

meta_df = raw_meta_df.copy()
meta_df["true_class"] = meta_df.apply(lookup_true_class, axis=1)

# Consistency checks between spectra and ground truth.
no_gt_row   = meta_df[~meta_df["object_stem"].isin(gt_indexed.index)]
no_class    = meta_df[meta_df["object_stem"].isin(gt_indexed.index) & meta_df["true_class"].isna()]
gt_no_spec  = set(gt_indexed.index) - set(meta_df["object_stem"])

print(f"\nSpectra whose object is absent from ground truth: {len(no_gt_row)}")
for n in no_gt_row["filename"].tolist()[:10]:
    print(f"  - {n}")
print(f"Spectra whose region has empty ground-truth class (excluded): {len(no_class)}")
for n in no_class["filename"].tolist()[:10]:
    print(f"  - {n}  (region={no_class.loc[no_class['filename'] == n, 'region'].iloc[0]})")
print(f"Ground-truth objects with no spectra at all: {len(gt_no_spec)}")
for s in sorted(gt_no_spec)[:10]:
    print(f"  - {s}")

# Drop spectra without a ground-truth class (one row per non-empty GT entry remains).
meta_df = meta_df[meta_df["true_class"].notna()].reset_index(drop=True)

# --- (c) Outer split ---
# Objects in object_split.csv keep their segmentation cv/test assignment; other
# objects go to "test" if listed in NEW_TEST_OBJECTS (Config), else to "cv".
split_df = pd.read_csv(OBJECT_SPLIT_CSV, dtype=str)
split_df.columns = [c.strip() for c in split_df.columns]
split_df["material"]    = split_df["material"].str.upper()
split_df["object_id"]   = split_df["object_id"].str.extract(r"(\d+)")[0].str.zfill(OBJECT_ID_WIDTH)
split_df["object_stem"] = split_df["material"] + "_obj_" + split_df["object_id"]
split_df["split"]       = split_df["split"].str.strip().str.lower()

print("\nobject_split.csv split counts (original segmentation split):")
display(split_df.groupby("split")["object_stem"].count().rename("n_objects"))

# Carry-over lookup and forced-test set for objects not in the segmentation split.
carryover_split = dict(zip(split_df["object_stem"], split_df["split"]))
NEW_TEST_STEMS  = {
    f"{mat.upper()}_obj_{int(oid):0{OBJECT_ID_WIDTH}d}" for mat, oid in NEW_TEST_OBJECTS
}

def assign_outer_split(object_stem):
    """cv/test for an object: carry over from segmentation split, else new-object rule."""
    if object_stem in carryover_split:
        return carryover_split[object_stem], "carryover"
    if object_stem in NEW_TEST_STEMS:
        return "test", "new_test"
    return "cv", "new_cv"

_split_origin = meta_df["object_stem"].map(lambda s: assign_outer_split(s))
meta_df["split"]        = _split_origin.map(lambda t: t[0])
meta_df["split_origin"] = _split_origin.map(lambda t: t[1])

# Object-level provenance of the split assignment.
obj_origin = (
    meta_df[["object_stem", "split", "split_origin"]]
    .drop_duplicates()
    .groupby(["split_origin", "split"]).size().rename("n_objects").reset_index()
)
print("\nOuter-split provenance (objects):")
display(obj_origin)

# Warn if a NEW_TEST_OBJECTS entry has no active spectra.
_missing_new_test = sorted(NEW_TEST_STEMS - set(meta_df["object_stem"]))
if _missing_new_test:
    print(f"\nWARNING: NEW_TEST_OBJECTS not found among active spectra: {_missing_new_test}")

# Keep only cv/test rows.
n_before = len(meta_df)
meta_df  = meta_df[meta_df["split"].isin(VALID_SPLITS)].reset_index(drop=True)
if n_before != len(meta_df):
    print(f"\nDropped {n_before - len(meta_df)} spectra outside {VALID_SPLITS}.")

# --- (d) Summaries and sample table export ---
ALL_CLASSES = sorted(meta_df["true_class"].unique())
print(f"\nFinal target classes ({len(ALL_CLASSES)}): {ALL_CLASSES}")

print("\nClass distribution per split:")
display(meta_df.groupby(["split", "true_class"]).size().rename("n_spectra").reset_index())

summary_rows = []
for split_val in ["cv", "test"]:
    sub = meta_df[meta_df["split"] == split_val]
    for region in VALID_REGIONS:
        rsub = sub[sub["region"] == region]
        if len(rsub):
            summary_rows.append({"split": split_val, "region": region,
                                 "n_objects": rsub["group_id"].nunique(), "n_spectra": len(rsub)})
    summary_rows.append({"split": split_val, "region": "ALL",
                         "n_objects": sub["group_id"].nunique(), "n_spectra": len(sub)})
split_summary_df = pd.DataFrame(summary_rows)
print("\nSplit summary (objects and spectra):")
display(split_summary_df)

meta_df.to_csv(SAMPLE_TABLE_CSV, index=False)
split_summary_df.to_csv(SPLIT_SUMMARY_CSV, index=False)
print(f"\nSaved sample table ({len(meta_df)} rows) to: {SAMPLE_TABLE_CSV}")
print(f"Saved split summary to: {SPLIT_SUMMARY_CSV}")


In [ ]:
# === Cell 8: Feature Loading ===

# Load each spectrum into a feature vector; label y = region-specific true_class.
# All spectra must share one wavenumber axis (taken from the first file).

def load_feature_vector(path):
    """Return (reflectance_vector, wavenumber_axis) for one spectrum file."""
    df = read_spectrum_file(path)
    return df["reflectance"].to_numpy(dtype=np.float64), df["wavenumber"].to_numpy()

print("Loading spectra ...")
feature_rows, wavenumber_ref, load_errors, n_features = [], None, [], None
for _, row in meta_df.iterrows():
    try:
        vec, wn = load_feature_vector(row["path"])
        if wavenumber_ref is None:
            wavenumber_ref, n_features = wn, len(vec)
            print(f"  Reference axis: {len(wn)} points, wn=[{wn.min():.1f}, {wn.max():.1f}] cm-1")
        if len(vec) != n_features:
            raise ValueError(f"length {len(vec)} != reference {n_features}")
        feature_rows.append(vec)
    except Exception as exc:
        load_errors.append({"filename": row["filename"], "error": str(exc)})

if load_errors:
    print(f"\nWARNING: {len(load_errors)} files failed to load:")
    for e in load_errors:
        print(f"  - {e['filename']}: {e['error']}")
if not feature_rows:
    raise RuntimeError("No spectra could be loaded.")

X_all = np.vstack(feature_rows)
y_all = meta_df["true_class"].to_numpy()
print(f"\nFeature matrix: {X_all.shape}  ({X_all.shape[0]} spectra x {X_all.shape[1]} points)")
print("Class counts:")
for cls in ALL_CLASSES:
    print(f"  {cls:6s}: {(y_all == cls).sum()}")

# Outer split masks
cv_mask, test_mask = (meta_df["split"] == "cv").to_numpy(), (meta_df["split"] == "test").to_numpy()
X_cv,   y_cv   = X_all[cv_mask],   y_all[cv_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]
meta_cv   = meta_df[cv_mask].reset_index(drop=True)
meta_test = meta_df[test_mask].reset_index(drop=True)

print(f"\nDevelopment (CV) set: {X_cv.shape[0]} spectra, {meta_cv['group_id'].nunique()} objects")
print(f"Held-out test set:    {X_test.shape[0]} spectra, {meta_test['group_id'].nunique()} objects")

# PCA components must not exceed the smallest fold's training-sample count.
max_pca = max([c for c in PCA_N_COMPONENTS_GRID if c is not None], default=0)
approx_min_train = int(X_cv.shape[0] * (NUM_FOLDS - 1) / NUM_FOLDS)
if max_pca >= approx_min_train:
    print(f"\nNOTE: largest PCA n_components ({max_pca}) is close to the approx. "
          f"per-fold training size (~{approx_min_train}). If PCA raises a size error, "
          f"lower PCA_N_COMPONENTS_GRID in Cell 4.")

In [ ]:
# === Cell 9: CV Fold Construction ===

# Stage A assigns whole objects to NUM_FOLDS folds (material-stratified); stage B
# moves individual spectra so each class has an even per-fold count, which can
# split objects across folds (logged to CSV). The test split stays object-level.

random.seed(RANDOM_SEED)

# --- Stage A: material-stratified object-level folds ---
cv_objects_df = meta_cv[["group_id", "prefix_material"]].drop_duplicates().reset_index(drop=True)
materials     = sorted(cv_objects_df["prefix_material"].unique())

init_fold_of_object = {}
for material in materials:
    obj_ids = sorted(cv_objects_df.loc[cv_objects_df["prefix_material"] == material, "group_id"])
    random.shuffle(obj_ids)
    sizes = [len(obj_ids) // NUM_FOLDS] * NUM_FOLDS
    for i in range(len(obj_ids) % NUM_FOLDS):
        sizes[i] += 1
    start = 0
    for fold_idx, sz in enumerate(sizes):
        for g in obj_ids[start:start + sz]:
            init_fold_of_object[g] = fold_idx
        start += sz

meta_cv = meta_cv.copy()
meta_cv["init_fold"] = meta_cv["group_id"].map(init_fold_of_object)
meta_cv["fold"]      = meta_cv["init_fold"]

def _per_class_fold_counts(df):
    return (df.groupby(["true_class", "fold"]).size()
              .unstack(fill_value=0)
              .reindex(columns=range(NUM_FOLDS), fill_value=0))

print("Per-class spectrum counts per fold BEFORE balancing (object-level only):")
display(_per_class_fold_counts(meta_cv))

# --- Stage B: greedy per-class balancing (deterministic) ---
def _fold_targets(total):
    """Even split of `total` over NUM_FOLDS; remainder to the lowest fold indices."""
    base, rem = divmod(total, NUM_FOLDS)
    return [base + (1 if f < rem else 0) for f in range(NUM_FOLDS)]

reassign_rows = []
for cls in sorted(meta_cv["true_class"].unique()):
    cls_index = meta_cv.index[meta_cv["true_class"] == cls].tolist()
    targets   = _fold_targets(len(cls_index))
    # Deterministic candidate order for which spectrum to move.
    ordered   = meta_cv.loc[cls_index].sort_values(["group_id", "region"]).index.tolist()

    while True:
        counts = [int((meta_cv.loc[ordered, "fold"] == f).sum()) for f in range(NUM_FOLDS)]
        over   = [f for f in range(NUM_FOLDS) if counts[f] > targets[f]]
        under  = [f for f in range(NUM_FOLDS) if counts[f] < targets[f]]
        if not over:
            break
        src, dst = over[0], under[0]
        # Move one spectrum of this class from an over-full to an under-full fold.
        move_idx = next(i for i in ordered if meta_cv.loc[i, "fold"] == src)
        meta_cv.loc[move_idx, "fold"] = dst
        reassign_rows.append({
            "object_stem": meta_cv.loc[move_idx, "object_stem"],
            "filename":    meta_cv.loc[move_idx, "filename"],
            "region":      meta_cv.loc[move_idx, "region"],
            "true_class":  cls,
            "from_fold":   src,
            "to_fold":     dst,
        })

print("\nPer-class spectrum counts per fold AFTER balancing:")
display(_per_class_fold_counts(meta_cv))

# --- Report reassignments and split objects ---
reassign_df = pd.DataFrame(
    reassign_rows,
    columns=["object_stem", "filename", "region", "true_class", "from_fold", "to_fold"],
)
reassign_df.to_csv(FOLD_REASSIGN_CSV, index=False)
print(f"\nSpectra reassigned by the balancing pass: {len(reassign_df)}")
if len(reassign_df):
    display(reassign_df)

# Objects whose spectra span more than one final fold.
obj_fold_spread = (
    meta_cv.groupby("group_id")
    .agg(folds=("fold", lambda s: sorted(set(int(x) for x in s))),
         n_spectra=("fold", "size"))
)
split_objects_df = obj_fold_spread[obj_fold_spread["folds"].map(len) > 1].reset_index()
split_objects_df["folds"] = split_objects_df["folds"].map(lambda fs: ",".join(map(str, fs)))
split_objects_df.to_csv(SPLIT_OBJECTS_CSV, index=False)
print(f"\nObjects split across folds by the balancing pass: {len(split_objects_df)} "
      f"of {meta_cv['group_id'].nunique()} CV objects")
if len(split_objects_df):
    display(split_objects_df)
print(f"Reassignment log saved to: {FOLD_REASSIGN_CSV}")
print(f"Split-object report saved to: {SPLIT_OBJECTS_CSV}")

# --- Save fold assignment and per-fold summary ---
fold_assignment_df = (
    meta_cv[["filename", "object_stem", "group_id", "prefix_material",
             "region", "true_class", "init_fold", "fold"]]
    .assign(reassigned=lambda d: d["init_fold"] != d["fold"])
    .sort_values(["fold", "true_class", "object_stem", "region"])
    .reset_index(drop=True)
)
fold_assignment_df.to_csv(FOLD_ASSIGNMENT_CSV, index=False)

fold_summary_rows = []
for fold_idx in range(NUM_FOLDS):
    val = meta_cv[meta_cv["fold"] == fold_idx]
    trn = meta_cv[meta_cv["fold"] != fold_idx]
    fold_summary_rows.append({
        "fold": fold_idx,
        "train_spectra": len(trn), "val_spectra": len(val),
        "val_objects":   val["group_id"].nunique(),
        "val_objects_also_in_train": int(
            val["group_id"].isin(trn["group_id"]).groupby(val["group_id"]).any().sum()
        ),
    })
print("\nPer-fold development split summary (spectrum-level folds):")
display(pd.DataFrame(fold_summary_rows))
print(f"Fold assignment table saved to: {FOLD_ASSIGNMENT_CSV}")

# --- sklearn (train_idx, val_idx) pairs over meta_cv / X_cv rows ---
CV_FOLD_INDICES = []
for fold_idx in range(NUM_FOLDS):
    val_mask = (meta_cv["fold"] == fold_idx).to_numpy()
    trn_mask = ~val_mask
    CV_FOLD_INDICES.append((np.where(trn_mask)[0], np.where(val_mask)[0]))
print("CV fold indices ready for GridSearchCV.")


In [ ]:
# === Cell 10: Hyperparameter Tuning and CV Evaluation ===

# Per classifier: Pipeline([StandardScaler, PCA, clf]) tuned by GridSearchCV on
# CV_FOLD_INDICES (PCA dims included), multi-metric scoring, refit on macro-F1.
# Per-fold metrics of the best candidate are read from cv_results_.

def compute_metrics(y_true, y_pred):
    """Primary and secondary classification metrics for one prediction set."""
    return {
        "macro-F1":          f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro-precision":   precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro-recall":      recall_score(y_true, y_pred, average="macro", zero_division=0),
        "accuracy":          accuracy_score(y_true, y_pred),
    }

def build_pipeline(estimator):
    """Build mean-centre -> PCA -> classifier pipeline.

    Spectra are already z-score normalised per sample, so features are only
    mean-centred (with_std=False) and not scaled to unit variance.
    """
    return Pipeline([
        ("scaler", StandardScaler(with_std=False)),   # mean-centre only, no variance scaling
        ("pca",    PCA(random_state=RANDOM_SEED)),
        ("clf",    estimator),
    ])

SCORING = {
    "macro-F1": "f1_macro", "balanced_accuracy": "balanced_accuracy",
    "macro-precision": "precision_macro", "macro-recall": "recall_macro",
    "accuracy": "accuracy",
}
METRIC_KEYS = list(SCORING.keys())

# Save parameter grids as JSON.
def _serialise_grid(grid):
    """Convert a param grid (dict or list of dicts) to a JSON-safe structure."""
    def _val(v):
        if v is None or isinstance(v, bool) or isinstance(v, (int, float)):
            return v
        return str(v)
    def _conv(d):
        return {k: [_val(x) for x in vs] for k, vs in d.items()}
    return [_conv(g) for g in grid] if isinstance(grid, list) else _conv(grid)

with PARAM_GRIDS_JSON.open("w", encoding="utf-8") as f:
    json.dump({n: _serialise_grid(g) for n, (_, g) in PARAM_GRIDS.items()},
              f, indent=2, ensure_ascii=False)
print(f"Hyperparameter grids saved to: {PARAM_GRIDS_JSON}\n")

hpo_results, cv_fold_rows, BEST_ESTIMATORS = [], [], {}

for clf_name, (estimator, param_grid) in PARAM_GRIDS.items():
    print("=" * 70)
    print(f"Classifier: {clf_name}")
    gs = GridSearchCV(
        estimator  = build_pipeline(estimator),
        param_grid = param_grid,
        cv         = CV_FOLD_INDICES,
        scoring    = SCORING,
        refit      = "macro-F1",
        n_jobs     = N_JOBS,
        verbose    = 0,
    )
    gs.fit(X_cv, y_cv)
    BEST_ESTIMATORS[clf_name] = gs.best_estimator_

    print(f"  Best params:      {gs.best_params_}")
    print(f"  Best CV macro-F1: {gs.best_score_:.4f}")

    hpo_results.append({
        "classifier":       clf_name,
        "best_cv_macro_f1": gs.best_score_,
        "best_n_components": gs.best_params_.get("pca__n_components", None),
        "best_params":      str(gs.best_params_),
    })

    best_idx = gs.best_index_
    for fold_idx in range(NUM_FOLDS):
        row = {"classifier": clf_name, "fold": fold_idx}
        for metric in METRIC_KEYS:
            row[metric] = gs.cv_results_[f"split{fold_idx}_test_{metric}"][best_idx]
        cv_fold_rows.append(row)
    print()

cv_fold_df = pd.DataFrame(cv_fold_rows)
agg_rows = []
for clf_name in PARAM_GRIDS:
    sub = cv_fold_df[cv_fold_df["classifier"] == clf_name]
    row = {"classifier": clf_name}
    for m in METRIC_KEYS:
        row[f"{m}_mean"], row[f"{m}_std"] = sub[m].mean(), sub[m].std(ddof=1)
    agg_rows.append(row)
cv_summary_df = pd.DataFrame(agg_rows)

cv_display = cv_summary_df[["classifier"]].copy()
for m in METRIC_KEYS:
    cv_display[m] = cv_summary_df.apply(lambda r: f"{r[f'{m}_mean']:.3f} +/- {r[f'{m}_std']:.3f}", axis=1)
print("CV performance summary (mean +/- std over folds):")
display(cv_display)

hpo_df = pd.DataFrame(hpo_results)
print("\nHPO best parameters and best CV macro-F1:")
display(hpo_df[["classifier", "best_cv_macro_f1", "best_n_components", "best_params"]])

hpo_df.to_csv(HPO_RESULTS_CSV, index=False)
cv_fold_df.to_csv(CV_METRICS_CSV, index=False)
print(f"\nHPO results saved to:     {HPO_RESULTS_CSV}")
print(f"CV fold metrics saved to: {CV_METRICS_CSV}")

# --- Per-class CV metrics per fold ---
# cv_results_ holds only aggregate scores, so each tuned pipeline is cloned, refit
# per fold and scored per class. Model selection is unaffected.
print("\nPer-class CV pass (refitting tuned pipelines per fold) ...")
cv_per_class_rows = []
for clf_name, best_pipe in BEST_ESTIMATORS.items():
    for fold_idx, (trn_idx, val_idx) in enumerate(CV_FOLD_INDICES):
        pipe = clone(best_pipe)
        pipe.fit(X_cv[trn_idx], y_cv[trn_idx])
        y_hat = pipe.predict(X_cv[val_idx])
        f1_per_class = f1_score(y_cv[val_idx], y_hat, labels=ALL_CLASSES,
                                average=None, zero_division=0)
        support = pd.Series(y_cv[val_idx]).value_counts()
        for cls, f1v in zip(ALL_CLASSES, f1_per_class):
            cv_per_class_rows.append({
                "classifier": clf_name, "fold": fold_idx, "class": cls,
                "f1": float(f1v), "support": int(support.get(cls, 0)),
            })

cv_per_class_df = pd.DataFrame(cv_per_class_rows)

# Per-class F1 summary across folds (mean +/- std).
cv_per_class_summary = (
    cv_per_class_df.groupby(["classifier", "class"])
    .agg(f1_mean=("f1", "mean"), f1_std=("f1", lambda s: s.std(ddof=1)),
         support_mean=("support", "mean"))
    .reset_index()
)
print("\nPer-class CV F1 (mean +/- std over folds):")
pc_display = cv_per_class_summary.copy()
pc_display["F1"] = pc_display.apply(lambda r: f"{r['f1_mean']:.3f} +/- {r['f1_std']:.3f}", axis=1)
display(pc_display.pivot(index="classifier", columns="class", values="F1"))

cv_per_class_df.to_csv(CV_PER_CLASS_CSV, index=False)
print(f"\nPer-class CV metrics saved to: {CV_PER_CLASS_CSV}")


In [ ]:
# === Cell 11: Final Test Evaluation ===

# BEST_ESTIMATORS are refit on the full CV pool by GridSearchCV;
# each is applied once to the held-out test set.

test_metric_rows, test_pred_rows = [], []
print("Test evaluation (single pass per classifier)\n" + "-" * 70)

for clf_name, best_pipe in BEST_ESTIMATORS.items():
    y_pred  = best_pipe.predict(X_test)
    y_proba = best_pipe.predict_proba(X_test) if hasattr(best_pipe, "predict_proba") else None

    metrics = compute_metrics(y_test, y_pred)
    test_metric_rows.append({"classifier": clf_name, **metrics})
    print(f"{clf_name}")
    for k, v in metrics.items():
        print(f"  {k:<22} {v:.4f}")
    print()

    for i, (_, row) in enumerate(meta_test.iterrows()):
        pred_row = {
            "classifier": clf_name, "filename": row["filename"], "group_id": row["group_id"],
            "prefix_material": row["prefix_material"], "region": row["region"],
            "y_true": y_test[i], "y_pred": y_pred[i], "correct": y_test[i] == y_pred[i],
        }
        if y_proba is not None:
            for j, cls in enumerate(best_pipe.classes_):
                pred_row[f"prob_{cls}"] = round(float(y_proba[i, j]), 6)
        test_pred_rows.append(pred_row)

test_metrics_df = pd.DataFrame(test_metric_rows)
print("=" * 70)
print("Test set comparison (held-out objects, single evaluation):")
display(test_metrics_df.round(4))

test_preds_df = pd.DataFrame(test_pred_rows)
print("\nRegion-wise performance per classifier:")
region_df = (
    test_preds_df.groupby(["classifier", "region"])
    .apply(lambda g: pd.Series({
        "n_spectra": len(g),
        "macro-F1":  f1_score(g["y_true"], g["y_pred"], average="macro", zero_division=0),
        "accuracy":  accuracy_score(g["y_true"], g["y_pred"]),
    }), include_groups=False)
    .reset_index()
)
display(region_df.round(4))

test_metrics_df.to_csv(TEST_METRICS_CSV, index=False)
test_preds_df.to_csv(TEST_PREDICTIONS_CSV, index=False)
print(f"\nTest metrics saved to:     {TEST_METRICS_CSV}")
print(f"Test predictions saved to: {TEST_PREDICTIONS_CSV}")

for clf_name, pipe in BEST_ESTIMATORS.items():
    mp = MODELS_DIR / f"{PROJECT_NAME}_{clf_name}_best_estimator.joblib"
    joblib.dump(pipe, mp)
    print(f"Saved {clf_name} estimator to: {mp}")

In [ ]:
# === Cell 12: Result Plots and Decision Boundaries ===

from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.base import clone
from matplotlib.colors import ListedColormap, BoundaryNorm

# --- 1. CV macro-F1 comparison (mean +/- std) ---
cv_agg = pd.DataFrame([{"classifier": r["classifier"], "mean": r["macro-F1_mean"],
                        "std": r["macro-F1_std"]} for r in agg_rows])
fig_cv, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(cv_agg))
ax.bar(x, cv_agg["mean"], yerr=cv_agg["std"], color="tab:blue", alpha=0.85, capsize=5, width=0.5)
ax.set_xticks(x); ax.set_xticklabels(cv_agg["classifier"]); ax.set_ylim(0, 1.05)
ax.set_ylabel("macro-F1"); ax.grid(axis="y", alpha=0.3)
ax.set_title("CV development performance - macro-F1 (mean +/- std, class-balanced folds)")
for xi, mv in zip(x, cv_agg["mean"]):
    ax.text(xi, mv + 0.01, f"{mv:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); fig_cv.savefig(CV_COMPARISON_PNG, bbox_inches="tight", dpi=150); plt.show()
print("Saved:", CV_COMPARISON_PNG)

# --- 2. Test macro-F1 comparison ---
fig_test, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(test_metrics_df))
ax.bar(x, test_metrics_df["macro-F1"], color="tab:orange", alpha=0.85, width=0.5)
ax.set_xticks(x); ax.set_xticklabels(test_metrics_df["classifier"]); ax.set_ylim(0, 1.05)
ax.set_ylabel("macro-F1"); ax.grid(axis="y", alpha=0.3)
ax.set_title("Test set performance - macro-F1 (held-out objects)")
for xi, v in zip(x, test_metrics_df["macro-F1"]):
    ax.text(xi, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); fig_test.savefig(TEST_COMPARISON_PNG, bbox_inches="tight", dpi=150); plt.show()
print("Saved:", TEST_COMPARISON_PNG)

# --- 3. Confusion matrices (row-normalised), one panel per classifier ---
clf_names = list(BEST_ESTIMATORS.keys())
n_cols    = 3
n_rows    = (len(clf_names) + n_cols - 1) // n_cols
fig_cm, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 4.6 * n_rows))
axes = np.array(axes).reshape(-1)
for ax, clf_name in zip(axes, clf_names):
    preds = test_preds_df[test_preds_df["classifier"] == clf_name]
    cm = confusion_matrix(preds["y_true"], preds["y_pred"], labels=ALL_CLASSES, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", ax=ax, xticklabels=ALL_CLASSES, yticklabels=ALL_CLASSES,
                cmap="Blues", linewidths=0.5, vmin=0, vmax=1, cbar=False)
    f1v = test_metrics_df.loc[test_metrics_df["classifier"] == clf_name, "macro-F1"].iloc[0]
    ax.set_title(f"{clf_name} (macro-F1={f1v:.3f})", fontsize=11)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=45)
for ax in axes[len(clf_names):]:
    ax.axis("off")
fig_cm.suptitle("Confusion matrices - held-out test set (row-normalised)", fontsize=13, y=1.01)
plt.tight_layout(); fig_cm.savefig(CONFUSION_MATRIX_PNG, bbox_inches="tight", dpi=150); plt.show()
print("Saved:", CONFUSION_MATRIX_PNG)

# --- 4. Decision boundary plots (2D PCA projection) ---
# Spectra are mean-centred and projected to 2 PCs; each tuned classifier is refit
# on the 2D data. For visualisation only.

DECISION_BOUNDARY_PNG = RESULTS_DIR / f"{PROJECT_NAME}_decision_boundaries.png"

# --- 4a. Project to 2D PCA ---
scaler_2d = StandardScaler(with_std=False).fit(X_cv)
pca_2d    = PCA(n_components=2, random_state=RANDOM_SEED)
X_cv_2d   = pca_2d.fit_transform(scaler_2d.transform(X_cv))
X_test_2d = pca_2d.transform(scaler_2d.transform(X_test))

ev = pca_2d.explained_variance_ratio_
print(f"\nDecision boundary visualisation — 2D PCA projection")
print(f"  Explained variance: PC1={ev[0]:.3f}, PC2={ev[1]:.3f}, total={ev.sum():.3f}")

# --- 4b. Colour setup (one colour per class, consistent across all panels) ---
n_classes    = len(ALL_CLASSES)
class_colors = [plt.cm.tab10(i) for i in range(n_classes)]
cm_bg        = ListedColormap(class_colors)                     # decision regions
cm_bright    = ListedColormap(class_colors)                     # scatter points

# Integer-encode labels for scatter c= argument (same order as sorted classes)
label_to_int = {c: i for i, c in enumerate(ALL_CLASSES)}
y_cv_int     = np.array([label_to_int[c] for c in y_cv])
y_test_int   = np.array([label_to_int[c] for c in y_test])

# --- 4c. One subplot per classifier ---
n_cols_db = 3
n_rows_db = (len(clf_names) + n_cols_db - 1) // n_cols_db
fig_db, axes_db = plt.subplots(n_rows_db, n_cols_db,
                               figsize=(6 * n_cols_db, 5.5 * n_rows_db))
axes_db = np.array(axes_db).reshape(-1)

# Axis limits with margin
x_min = min(X_cv_2d[:, 0].min(), X_test_2d[:, 0].min()) - 0.5
x_max = max(X_cv_2d[:, 0].max(), X_test_2d[:, 0].max()) + 0.5
y_min = min(X_cv_2d[:, 1].min(), X_test_2d[:, 1].min()) - 0.5
y_max = max(X_cv_2d[:, 1].max(), X_test_2d[:, 1].max()) + 0.5

for ax, clf_name in zip(axes_db, clf_names):
    # Clone the tuned classifier step and refit it on the 2D data.
    best_clf_step = BEST_ESTIMATORS[clf_name].named_steps["clf"]
    clf_2d = clone(best_clf_step)
    clf_2d.fit(X_cv_2d, y_cv)

    # Decision regions on a meshgrid; string predictions are mapped via
    # label_to_int so fill colours match the scatter colours.
    _res  = 200
    _xx, _yy = np.meshgrid(np.linspace(x_min, x_max, _res),
                        np.linspace(y_min, y_max, _res))
    _Z = np.array([label_to_int[c]
               for c in clf_2d.predict(np.c_[_xx.ravel(), _yy.ravel()])])
    _Z = _Z.reshape(_xx.shape)
    _norm = BoundaryNorm(np.arange(-0.5, n_classes, 1.0), n_classes)
    ax.pcolormesh(_xx, _yy, _Z, cmap=cm_bg, norm=_norm, alpha=0.35, zorder=1)
    
    
    

    # Training points (solid)
    ax.scatter(X_cv_2d[:, 0], X_cv_2d[:, 1],
               c=y_cv_int, cmap=cm_bright,
               edgecolors="k", s=30, linewidths=0.5,
               vmin=0, vmax=n_classes - 1, zorder=3)
    # Test points (diamonds, semi-transparent)
    ax.scatter(X_test_2d[:, 0], X_test_2d[:, 1],
               c=y_test_int, cmap=cm_bright,
               edgecolors="k", s=55, linewidths=0.5,
               alpha=0.6, marker="D",
               vmin=0, vmax=n_classes - 1, zorder=4)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    # 2D and full-dimensional test macro-F1 in the title
    score_2d = f1_score(y_test, clf_2d.predict(X_test_2d),
                        average="macro", zero_division=0)
    f1_full  = test_metrics_df.loc[
        test_metrics_df["classifier"] == clf_name, "macro-F1"
    ].iloc[0]
    ax.set_title(f"{clf_name}\n2D F1={score_2d:.3f} | full F1={f1_full:.3f}", fontsize=11)
    ax.set_xlabel(f"PC 1 ({ev[0]:.1%} var)")
    ax.set_ylabel(f"PC 2 ({ev[1]:.1%} var)")

# Turn off unused subplots
for ax in axes_db[len(clf_names):]:
    ax.axis("off")

# Shared legend below the figure
legend_handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=class_colors[i], markersize=8,
               markeredgecolor="k", markeredgewidth=0.5,
               label=ALL_CLASSES[i])
    for i in range(n_classes)
]
legend_handles.append(
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor="grey", markersize=8,
               markeredgecolor="k", markeredgewidth=0.5,
               label="Train (CV pool)")
)
legend_handles.append(
    plt.Line2D([0], [0], marker="D", color="w",
               markerfacecolor="grey", markersize=8,
               markeredgecolor="k", markeredgewidth=0.5,
               alpha=0.6, label="Test (held-out)")
)
fig_db.legend(handles=legend_handles, loc="lower center",
              ncol=min(n_classes + 2, 8), fontsize=9,
              bbox_to_anchor=(0.5, -0.04))

fig_db.suptitle(
    f"Decision boundaries — 2D PCA projection "
    f"(explained variance: {ev.sum():.1%})",
    fontsize=13, y=1.01,
)
plt.tight_layout()
fig_db.savefig(DECISION_BOUNDARY_PNG, bbox_inches="tight", dpi=150)
plt.show()
print("Saved:", DECISION_BOUNDARY_PNG)

# --- 5. Final ranking printout ---
print("\n" + "=" * 70 + "\nBENCHMARK COMPLETE\n" + "=" * 70)
print("\nCV macro-F1 (mean +/- std):")
for _, r in cv_summary_df.iterrows():
    print(f"  {r['classifier']:<5} {r['macro-F1_mean']:.3f} +/- {r['macro-F1_std']:.3f}")
print("\nTest macro-F1 (ranked):")
for _, r in test_metrics_df.sort_values("macro-F1", ascending=False).iterrows():
    print(f"  {r['classifier']:<5} F1={r['macro-F1']:.3f}  BalAcc={r['balanced_accuracy']:.3f}  Acc={r['accuracy']:.3f}")
best = test_metrics_df.sort_values("macro-F1", ascending=False).iloc[0]
print(f"\nBest test classifier: {best['classifier']} (macro-F1={best['macro-F1']:.3f})")
print("\nOutputs written to:", RESULTS_DIR)

In [ ]:
# === Cell 13: Preprocessing Ablation Plots ===
# Standalone. Reads the results of all preprocessing configurations (C0..Cn, one run
# of this notebook each) and plots CV/test macro-F1, test Brier Score and AUC-ROC per
# classifier plus the mean over classifiers (error bars: std); exports the plot data.

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

# --- Config — paths ---
ABLATION_ROOT = Path(
    f"{KI_SORT_ROOT}/KI_Sort_Project/05_IR_Classification/"
    "20260615_Classifier_Benchmark_Preprocessing_Influence"
)
ABLATION_RESULTS_DIR = ABLATION_ROOT / "results"   # plots are written here
ABLATION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_NAME = ABLATION_ROOT.name                  # output filename prefix

PRIMARY_METRIC = "macro-F1"                          # column read from both CSVs

# Per-configuration result folders are matched by this pattern; the captured
# group ("C0", "C1", ...) is used as the short x-axis label.
CONFIG_DIR_REGEX = re.compile(r"^(C\d+)")

# Canonical classifier order (any others present are appended alphabetically).
CLASSIFIER_ORDER = ["SVM", "kNN", "RF", "DT", "LR"]

# --- Config — appearance (adjustable) ---
PALETTE            = "tab10"     # seaborn/matplotlib palette name OR list of colours
MEAN_DOT_COLOR     = "black"
CLASSIFIER_MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]  # cycled per classifier
MEAN_MARKER        = "o"
FONT_FAMILY        = "DejaVu Sans"
FONT_SIZE          = 10          # 10 pt at 16 cm LaTeX text width
FIGSIZE            = (16 / 2.54, 3.65)  # 16 cm wide, in inches
DOT_SIZE           = 70          # scatter `s` parameter (area, not diameter)
MEAN_DOT_SIZE      = 90
DODGE_TOTAL        = 0.50        # total horizontal spread within one config slot
ERRORBAR_CAPSIZE   = 3
ERRORBAR_LW        = 1.2
LABEL_FONTSIZE     = 8           # mean-value annotations
XTICK_ROTATION     = 0
BAND_COLOR         = "#f0f0f0"   # light grey for alternating background bands

# Font settings.
plt.rcParams["font.family"]     = FONT_FAMILY
plt.rcParams["font.size"]       = FONT_SIZE
plt.rcParams["axes.titlesize"]  = FONT_SIZE + 1
plt.rcParams["axes.labelsize"]  = FONT_SIZE
plt.rcParams["legend.fontsize"] = FONT_SIZE - 1
plt.rcParams["savefig.dpi"]     = 150


# --- 1. Discover configuration result folders ---
def _discover_configs(root: Path):
    """Return [(label, config_dir_name, results_dir), ...] sorted by label."""
    found = []
    for d in sorted(p for p in root.iterdir() if p.is_dir()):
        m = CONFIG_DIR_REGEX.match(d.name)
        if not m:
            continue
        if d.name == ABLATION_RESULTS_DIR.name:        # skip the output dir
            continue
        res = d / "results"
        if not res.is_dir():
            print(f"  [skip] {d.name}: no results/ subdirectory")
            continue
        found.append((m.group(1), d.name, res))
    return found


configs = _discover_configs(ABLATION_ROOT)
if not configs:
    raise RuntimeError(f"No 'C<n>_*' configuration folders found under {ABLATION_ROOT}")
print(f"Discovered {len(configs)} configurations: {[c[0] for c in configs]}")


# --- 2. Load CV (per-fold) and test (single) macro-F1 for every config ---
cv_long_rows, test_long_rows = [], []
for label, cfg_name, res_dir in configs:
    cv_csv   = res_dir / f"{cfg_name}_cv_metrics.csv"
    test_csv = res_dir / f"{cfg_name}_test_metrics.csv"

    if not cv_csv.exists():
        print(f"  [warn] {label}: missing {cv_csv.name}")
    else:
        cv = pd.read_csv(cv_csv)
        # Per classifier: mean +/- std of the primary metric over folds.
        agg = (cv.groupby("classifier")[PRIMARY_METRIC]
                 .agg(mean="mean", std=lambda s: s.std(ddof=1))
                 .reset_index())
        for _, r in agg.iterrows():
            cv_long_rows.append({"config": label, "classifier": r["classifier"],
                                 "mean": r["mean"], "std": r["std"]})

    if not test_csv.exists():
        print(f"  [warn] {label}: missing {test_csv.name}")
    else:
        te = pd.read_csv(test_csv)
        for _, r in te.iterrows():
            test_long_rows.append({"config": label, "classifier": r["classifier"],
                                   "value": r[PRIMARY_METRIC]})

cv_long   = pd.DataFrame(cv_long_rows)
test_long = pd.DataFrame(test_long_rows)
if cv_long.empty and test_long.empty:
    raise RuntimeError("No CV or test metrics could be loaded; nothing to plot.")


# --- 2b. Load test predictions and compute Brier Score & AUC-ROC per config ---
# Computed from the prob_* columns of each configuration's test_predictions.csv
# (written by the Final Test Evaluation cell).

brier_long_rows, auc_long_rows = [], []
_proba_load_warnings = []

for label, cfg_name, res_dir in configs:
    preds_csv = res_dir / f"{cfg_name}_test_predictions.csv"
    if not preds_csv.exists():
        _proba_load_warnings.append(f"  [warn] {label}: missing {preds_csv.name}")
        continue

    preds = pd.read_csv(preds_csv)

    # Identify probability columns and derive the class list from them.
    prob_cols = sorted([c for c in preds.columns if c.startswith("prob_")])
    if not prob_cols:
        _proba_load_warnings.append(
            f"  [warn] {label}: {preds_csv.name} has no prob_* columns "
            f"(classifier may lack predict_proba)"
        )
        continue

    classes = [c.replace("prob_", "") for c in prob_cols]

    for clf_name, grp in preds.groupby("classifier"):
        # Skip classifiers without probability outputs.
        proba = grp[prob_cols].to_numpy(dtype=float)
        if np.isnan(proba).all():
            continue

        y_true  = grp["y_true"].to_numpy()
        y_ohe   = label_binarize(y_true, classes=classes)

        # Brier Score (multiclass): mean squared error between one-hot
        # truth and predicted probabilities.  Lower is better.
        brier = float(np.mean(np.sum((y_ohe - proba) ** 2, axis=1)))
        brier_long_rows.append({
            "config": label, "classifier": clf_name, "value": brier,
        })

        # AUC-ROC (macro OVR).  Requires >= 2 classes present in y_true.
        if len(set(y_true)) >= 2:
            try:
                auc = float(roc_auc_score(
                    y_ohe, proba, multi_class="ovr", average="macro",
                ))
            except ValueError:
                auc = np.nan
        else:
            auc = np.nan
        auc_long_rows.append({
            "config": label, "classifier": clf_name, "value": auc,
        })

brier_long = pd.DataFrame(brier_long_rows)
auc_long   = pd.DataFrame(auc_long_rows)

if _proba_load_warnings:
    for w in _proba_load_warnings:
        print(w)

if not brier_long.empty:
    print(f"Brier Score computed for {brier_long['config'].nunique()} configs, "
          f"{brier_long['classifier'].nunique()} classifiers")
if not auc_long.empty:
    print(f"AUC-ROC computed for {auc_long['config'].nunique()} configs, "
          f"{auc_long['classifier'].nunique()} classifiers")


# --- 3. Shared style maps (colour and marker per classifier) ---
present = set(cv_long.get("classifier", pd.Series(dtype=str))) | \
          set(test_long.get("classifier", pd.Series(dtype=str)))
ordered_clfs = [c for c in CLASSIFIER_ORDER if c in present] + \
               sorted(c for c in present if c not in CLASSIFIER_ORDER)

if isinstance(PALETTE, (list, tuple)):
    base_colors = list(PALETTE)
else:
    base_colors = sns.color_palette(PALETTE, n_colors=max(len(ordered_clfs), 1))
clf_color  = {c: base_colors[i % len(base_colors)] for i, c in enumerate(ordered_clfs)}
clf_marker = {c: CLASSIFIER_MARKERS[i % len(CLASSIFIER_MARKERS)]
              for i, c in enumerate(ordered_clfs)}


def _plot_panel(long_df, value_col, std_col, title, ylabel, out_png,
                higher_is_better=True, legend_loc="lower right"):
    """Dodged dot plot — one dot per classifier per config, no connecting lines.

    Parameters
    ----------
    long_df : DataFrame   with columns [config, classifier, value_col (, std_col)].
    value_col : str       column name for the metric value (or "mean" for CV).
    std_col : str or None if not None, used as per-classifier error bars.
    higher_is_better : bool
        True  → sort configs ascending  (lowest mean on the left).
        False → sort configs descending (highest mean on the left).
    legend_loc : str      matplotlib legend location string.
    """
    # Wide tables: rows=config, cols=classifier
    wide_mean = long_df.pivot(index="config", columns="classifier", values=value_col)
    wide_std  = (long_df.pivot(index="config", columns="classifier", values=std_col)
                 if std_col else None)

    # Grand mean / std over classifiers per config; defines the x order.
    grand_mean = wide_mean.mean(axis=1)
    grand_std  = wide_mean.std(axis=1, ddof=1)
    order      = grand_mean.sort_values(ascending=higher_is_better).index.tolist()

    wide_mean  = wide_mean.loc[order]
    if wide_std is not None:
        wide_std = wide_std.loc[order]
    grand_mean = grand_mean.loc[order]
    grand_std  = grand_std.loc[order]

    # --- Dodge offsets (classifiers first, then mean) ---
    active_clfs = [c for c in ordered_clfs if c in wide_mean.columns]
    n_items     = len(active_clfs) + 1                  # classifiers + mean
    offsets     = np.linspace(-DODGE_TOTAL / 2, DODGE_TOTAL / 2, n_items)
    clf_offset  = {clf: offsets[i] for i, clf in enumerate(active_clfs)}
    mean_offset = offsets[-1]

    x   = np.arange(len(order))
    fig, ax = plt.subplots(figsize=FIGSIZE)

    # --- Alternating background bands (grey / white) per config slot ---
    for i in range(len(order)):
        if i % 2 == 0:
            ax.axvspan(i - 0.5, i + 0.5, color=BAND_COLOR, zorder=0)

    # --- Per-classifier dots (+ optional error bars for CV) ---
    for clf in active_clfs:
        y  = wide_mean[clf].to_numpy(dtype=float)
        xd = x + clf_offset[clf]

        if wide_std is not None:
            s = wide_std[clf].to_numpy(dtype=float)
            ax.errorbar(xd, y, yerr=s, fmt="none",
                        ecolor=clf_color[clf], elinewidth=ERRORBAR_LW,
                        capsize=ERRORBAR_CAPSIZE, alpha=0.55, zorder=2)

        ax.scatter(xd, y, color=clf_color[clf], marker=clf_marker[clf],
                   s=DOT_SIZE, edgecolors="black", linewidths=0.4,
                   label=clf, zorder=3)

    # --- Grand-mean dot with error bar + value label ---
    gm = grand_mean.to_numpy(dtype=float)
    gs = grand_std.to_numpy(dtype=float)
    xm = x + mean_offset

    ax.errorbar(xm, gm, yerr=gs, fmt="none",
                ecolor=MEAN_DOT_COLOR, elinewidth=ERRORBAR_LW + 0.4,
                capsize=ERRORBAR_CAPSIZE + 1, alpha=0.7, zorder=4)
    ax.scatter(xm, gm, color=MEAN_DOT_COLOR, marker=MEAN_MARKER,
               s=MEAN_DOT_SIZE, edgecolors="black", linewidths=0.5,
               label="Mean", zorder=5)

    # Value labels for the mean dot (offset to the right)
    for xi, val in zip(xm, gm):
        ax.annotate(f"{val:.3f}", xy=(xi, val),
                    xytext=(6, 0), textcoords="offset points",
                    fontsize=LABEL_FONTSIZE, va="center", ha="left",
                    color=MEAN_DOT_COLOR, fontweight="bold")

    # --- Axes / decorations ---
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=XTICK_ROTATION)
    sort_dir = "ascending mean" if higher_is_better else "descending mean"
    ax.set_xlabel(f"Preprocessing configuration ({sort_dir})")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc=legend_loc, framealpha=0.9, ncol=2)

    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", out_png)
    return out_png


# --- 4. Plot 1 — CV development macro-F1 (with per-classifier error bars) ---
if not cv_long.empty:
    _plot_panel(
        cv_long, value_col="mean", std_col="std",
        title=f"CV {PRIMARY_METRIC} Across Preprocessing Configurations",
        ylabel=PRIMARY_METRIC,
        out_png=ABLATION_RESULTS_DIR / f"{ABLATION_NAME}_cv_{PRIMARY_METRIC}_by_config.png",
        higher_is_better=True,
        legend_loc="lower right",
    )
else:
    print("No CV metrics loaded — skipping CV plot.")

# --- 5. Plot 2 — held-out test macro-F1 ---
if not test_long.empty:
    _plot_panel(
        test_long, value_col="value", std_col=None,
        title=f"Test {PRIMARY_METRIC} Across Preprocessing Configurations",
        ylabel=PRIMARY_METRIC,
        out_png=ABLATION_RESULTS_DIR / f"{ABLATION_NAME}_test_{PRIMARY_METRIC}_by_config.png",
        higher_is_better=True,
        legend_loc="lower right",
    )
else:
    print("No test metrics loaded — skipping test plot.")

# --- 5b. Plot 3 — held-out test Brier Score (lower is better → high to low) ---
if not brier_long.empty:
    _plot_panel(
        brier_long, value_col="value", std_col=None,
        title="Test Brier Score Across Preprocessing Configurations",
        ylabel="Brier Score",
        out_png=ABLATION_RESULTS_DIR / f"{ABLATION_NAME}_test_brier_score_by_config.png",
        higher_is_better=False,
        legend_loc="upper right",
    )
else:
    print("No Brier Score data — skipping Brier Score plot.")

# --- 5c. Plot 4 — held-out test AUC-ROC (higher is better) ---
if not auc_long.empty:
    _plot_panel(
        auc_long, value_col="value", std_col=None,
        title="Test AUC-ROC (Macro OVR) Across Preprocessing Configurations",
        ylabel="AUC-ROC",
        out_png=ABLATION_RESULTS_DIR / f"{ABLATION_NAME}_test_auc_roc_by_config.png",
        higher_is_better=True,
        legend_loc="lower right",
    )
else:
    print("No AUC-ROC data — skipping AUC-ROC plot.")


# --- 6. Export the exact values plotted above to CSV ---
# One CSV per panel with per-classifier and mean values, rows in x-axis order.

def _export_plot_data(long_df, value_col, std_col, metric_name, out_csv,
                      higher_is_better=True):
    wide_mean = long_df.pivot(index="config", columns="classifier", values=value_col)
    wide_std  = (long_df.pivot(index="config", columns="classifier", values=std_col)
                 if std_col else None)

    grand_mean = wide_mean.mean(axis=1)
    grand_std  = wide_mean.std(axis=1, ddof=1)
    order      = grand_mean.sort_values(ascending=higher_is_better).index.tolist()

    rows = []
    for cfg in order:
        for clf in ordered_clfs:
            if clf not in wide_mean.columns:
                continue
            rows.append({
                "config":     cfg,
                "metric":     metric_name,
                "classifier": clf,
                "mean":       wide_mean.loc[cfg, clf],
                "std":        (wide_std.loc[cfg, clf] if wide_std is not None else np.nan),
                "std_basis":  ("over_folds" if std_col else "none"),
            })
        rows.append({
            "config":     cfg,
            "metric":     metric_name,
            "classifier": "Mean",
            "mean":       grand_mean.loc[cfg],
            "std":        grand_std.loc[cfg],
            "std_basis":  "over_classifiers",
        })

    df = pd.DataFrame(rows, columns=["config", "metric", "classifier", "mean", "std", "std_basis"])
    df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    return df


if not cv_long.empty:
    _export_plot_data(
        cv_long, value_col="mean", std_col="std", metric_name=PRIMARY_METRIC,
        out_csv=ABLATION_RESULTS_DIR / f"{ABLATION_NAME}_cv_{PRIMARY_METRIC}_plotdata.csv",
        higher_is_better=True,
    )
if not test_long.empty:
    _export_plot_data(
        test_long, value_col="value", std_col=None, metric_name=PRIMARY_METRIC,
        out_csv=ABLATION_RESULTS_DIR / f"{ABLATION_NAME}_test_{PRIMARY_METRIC}_plotdata.csv",
        higher_is_better=True,
    )
if not brier_long.empty:
    _export_plot_data(
        brier_long, value_col="value", std_col=None, metric_name="brier_score",
        out_csv=ABLATION_RESULTS_DIR / f"{ABLATION_NAME}_test_brier_score_plotdata.csv",
        higher_is_better=False,
    )
if not auc_long.empty:
    _export_plot_data(
        auc_long, value_col="value", std_col=None, metric_name="AUC-ROC",
        out_csv=ABLATION_RESULTS_DIR / f"{ABLATION_NAME}_test_auc_roc_plotdata.csv",
        higher_is_better=True,
    )

In [ ]:
# === Cell 14: Mean-Only Ablation Plot ===
# Mean over classifiers per preprocessing configuration (ascending), with descriptive
# labels and fixed colours. Uses names from the Preprocessing Ablation Plots cell.

# --- Plot settings ---
_MO_SRC_LONG   = test_long          # source: test_long / cv_long / brier_long / auc_long
_MO_VALUE_COL  = "value"            # "value" for test/brier/auc, "mean" for cv_long
_MO_YLABEL     = PRIMARY_METRIC
_MO_TITLE      = f"Mean Test {PRIMARY_METRIC} Across Preprocessing Configurations"
_MO_OUT_PNG    = ABLATION_RESULTS_DIR / f"{ABLATION_NAME}_test_{PRIMARY_METRIC}_mean_only.png"
_MO_ROTATION   = 45                 # x tick label rotation (longer descriptive labels)

# Descriptive preprocessing names per config label.
_MO_LABEL_MAP = {
    "C0": "Raw",
    "C1": "Norm.",
    "C2": "BC Norm.",
    "C3": "Smooth. Norm.",
    "C4": "BC Smooth. Norm.",
    "C5": "1st Deriv.",
    "C6": "2nd Deriv.",
    "C7": "BC 1st Deriv.",
    "C8": "BC 2nd Deriv.",
}

# Fixed dot colour per config (RGB 0-1 tuples).
_MO_COLOR_MAP = {
    "C0": (0 / 255, 0 / 255, 0 / 255),          # black
    "C1": (151 / 255, 41 / 255, 130 / 255),
    "C2": (151 / 255, 41 / 255, 130 / 255),
    "C3": (90 / 255, 86 / 255, 161 / 255),
    "C4": (90 / 255, 86 / 255, 161 / 255),
    "C5": (138 / 255, 190 / 255, 231 / 255),
    "C6": (138 / 255, 190 / 255, 231 / 255),
    "C7": (138 / 255, 190 / 255, 231 / 255),
    "C8": (138 / 255, 190 / 255, 231 / 255),
}
_MO_DEFAULT_COLOR = "black"   # fallback for any config not in the map above

if _MO_SRC_LONG is not None and not _MO_SRC_LONG.empty:
    # Grand mean over classifiers, per config (same computation as _plot_panel).
    _mo_wide  = _MO_SRC_LONG.pivot(index="config", columns="classifier",
                                   values=_MO_VALUE_COL)
    _mo_grand = _mo_wide.mean(axis=1)

    # Ascending-mean order.
    _mo_order = _mo_grand.sort_values(ascending=True).index.tolist()

    _mo_grand      = _mo_grand.loc[_mo_order]
    _mo_x          = np.arange(len(_mo_order))
    _mo_y          = _mo_grand.to_numpy(dtype=float)
    _mo_ticklabels = [_MO_LABEL_MAP.get(c, c) for c in _mo_order]
    _mo_colors     = [_MO_COLOR_MAP.get(c, _MO_DEFAULT_COLOR) for c in _mo_order]

    fig, ax = plt.subplots(figsize=FIGSIZE)

    # Alternating background bands.
    for i in range(len(_mo_order)):
        if i % 2 == 0:
            ax.axvspan(i - 0.5, i + 0.5, color=BAND_COLOR, zorder=0)

    # Mean dots only — no per-classifier dots, no error bars.
    ax.scatter(_mo_x, _mo_y, color=_mo_colors, marker=MEAN_MARKER,
               s=MEAN_DOT_SIZE, edgecolors="black", linewidths=0.5, zorder=3)

    # Bold value labels above each dot, colored to match the dot.
    for xi, val, col in zip(_mo_x, _mo_y, _mo_colors):
        ax.annotate(f"{val:.3f}", xy=(xi, val),
                    xytext=(0, 5), textcoords="offset points",
                    fontsize=LABEL_FONTSIZE, va="bottom", ha="center",
                    color=col, fontweight="bold")

    # Headroom for the value labels (~18 % of the data span above the top).
    _mo_lo, _mo_hi = float(np.nanmin(_mo_y)), float(np.nanmax(_mo_y))
    _mo_pad = (_mo_hi - _mo_lo) * 0.18 if _mo_hi > _mo_lo else max(abs(_mo_hi), 1e-6) * 0.05
    ax.set_ylim(_mo_lo - _mo_pad * 0.4, _mo_hi + _mo_pad)

    ax.set_xticks(_mo_x)
    ax.set_xticklabels(_mo_ticklabels, rotation=_MO_ROTATION,
                       ha="right" if _MO_ROTATION else "center")
    ax.set_xlabel("Preprocessing Configuration")
    ax.set_ylabel(_MO_YLABEL)
    ax.set_title(_MO_TITLE)
    ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(_MO_OUT_PNG, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", _MO_OUT_PNG)

    # Export the exact values plotted.
    _mo_csv = ABLATION_RESULTS_DIR / f"{ABLATION_NAME}_test_{PRIMARY_METRIC}_mean_only_plotdata.csv"
    pd.DataFrame({
        "config":      _mo_order,
        "config_name": _mo_ticklabels,
        "metric":      _MO_YLABEL,
        "mean":        _mo_y,
    }).to_csv(_mo_csv, index=False)
    print("Saved:", _mo_csv)
else:
    print("Mean-only panel: source DataFrame is empty — skipping.")